In [1]:
import pandas as pd
import pyarrow.parquet as pq
import os
import numpy as np

In [2]:
# import final datasets

weather_final = pd.read_csv("combined_weather_final.csv")
cdc_final = pd.read_csv("combined_cdc_final.csv")
census_final = pd.read_csv("combined_census_final.csv")

In [3]:
# Standardise key columns across all three datasets
weather_final["CountyFIPS"] = weather_final["CountyFIPS"].astype(float).astype(int).astype(str).str.zfill(5)
cdc_final["CountyFIPS"] = cdc_final["CountyFIPS"].astype(str).str.zfill(5)
census_final["CountyFIPS"] = census_final["CountyFIPS"].astype(str).str.zfill(5)

# Extract year from DATE column in weather_final
weather_final["year"] = weather_final["DATE"].astype(int)
weather_final = weather_final.drop(columns=["DATE"])

# Rename year columns to match
cdc_final = cdc_final.rename(columns={"Year": "year"})
census_final = census_final.rename(columns={"year": "year"})

# Left merge cdc_final with weather data
merged_df = cdc_final.merge(
    weather_final,
    on=["year", "CountyFIPS"],
    how="left"
)

# Left merge with census data
merged_df = merged_df.merge(
    census_final,
    on=["year", "CountyFIPS"],
    how="left"
)

print(f"CDC final shape: {cdc_final.shape}")
print(f"Weather final shape: {weather_final.shape}")
print(f"Census final shape: {census_final.shape}")
print(f"Merged shape: {merged_df.shape}")
print(f"Unique counties: {merged_df['CountyFIPS'].nunique()}")
print(f"Unique years: {sorted(merged_df['year'].unique())}")
print(f"\nMissing values from weather merge: {merged_df['TAVG'].isna().sum()}")
print(f"Missing values from census merge: {merged_df['total_population'].isna().sum()}")
merged_df.head()

CDC final shape: (20462, 11)
Weather final shape: (11594, 27)
Census final shape: (35427, 17)
Merged shape: (20478, 51)
Unique counties: 3143
Unique years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]

Missing values from weather merge: 14190
Missing values from census merge: 0


,year,StateAbbr,County name,CountyFIPS,BPHIGH,CASTHMA,COPD,MHLTH,PHLTH,SLEEP,...,pct_bachelors_plus,pct_graduate_degree,pct_less_than_hs,pct_white,pct_black,pct_asian,pct_hispanic,median_household_income,state,county
0,2013,AK,ANCHORAGE MUNICIPALITY ...,02020,28.691667,NaN,NaN,NaN,NaN,NaN,...,11.7,92.5,23.2,0.1,7.9,7.8,92.0,77454,2,20
1,2013,AL,JEFFERSON COUNTY ...,01073,41.893750,NaN,NaN,NaN,NaN,NaN,...,11.6,87.4,26.6,0.1,0.8,0.8,96.2,45429,1,73
2,2013,AL,MADISON COUNTY ...,01089,37.654348,NaN,NaN,NaN,NaN,NaN,...,14.5,90.0,21.4,0.1,2.3,2.2,95.4,58434,1,89
3,2013,AL,MOBILE COUNTY ...,01097,44.833333,NaN,NaN,NaN,NaN,NaN,...,7.0,83.9,33.0,0.0,1.4,1.4,97.5,43028,1,97
4,2013,AL,MONTGOMERY COUNTY ...,01101,39.792727,NaN,NaN,NaN,NaN,NaN,...,12.0,85.6,26.2,0.1,1.3,1.1,96.5,44790,1,101


In [4]:
# Check year overlap before merging
print("CDC years:", sorted(cdc_final['year'].unique()))
print("Weather years:", sorted(weather_final['year'].unique()))
print("Census years:", sorted(census_final['year'].unique()))


CDC years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Weather years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Census years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


In [5]:
# Check year values and dtypes
print("CDC year dtype:", cdc_final['year'].dtype)
print("Weather year dtype:", weather_final['year'].dtype)
print("CDC years:", sorted(cdc_final['year'].unique()))
print("Weather years:", sorted(weather_final['year'].unique()))

# Check CountyFIPS values and dtypes
print("\nCDC CountyFIPS dtype:", cdc_final['CountyFIPS'].dtype)
print("Weather CountyFIPS dtype:", weather_final['CountyFIPS'].dtype)
print("\nSample CDC CountyFIPS:", cdc_final['CountyFIPS'].head(5).tolist())
print("Sample Weather CountyFIPS:", weather_final['CountyFIPS'].head(5).tolist())

# Check how many keys actually overlap
cdc_keys = set(zip(cdc_final['year'], cdc_final['CountyFIPS']))
weather_keys = set(zip(weather_final['year'], weather_final['CountyFIPS']))
print(f"\nCDC unique (year, CountyFIPS) pairs: {len(cdc_keys)}")
print(f"Weather unique (year, CountyFIPS) pairs: {len(weather_keys)}")
print(f"Overlapping pairs: {len(cdc_keys & weather_keys)}")

CDC year dtype: int64
Weather year dtype: int64
CDC years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Weather years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]

CDC CountyFIPS dtype: str
Weather CountyFIPS dtype: str

Sample CDC CountyFIPS: ['02020', '01073', '01089', '01097', '01101']
Sample Weather CountyFIPS: ['02130', '02130', '02130', '02130', '02130']

CDC unique (year, CountyFIPS) pairs: 20462
Weather unique (year, CountyFIPS) pairs: 11571
Overlapping pairs: 7080


In [6]:
# Define weather columns
weather_cols = ["CLDD", "DT32", "DX32", "DX70", "DX90", "EMNT", "EMXT", "HTDD", "PRCP", "TAVG", "TMAX", "TMIN"]

# Calculate missing rate per county across all years
county_missing = merged_df.groupby("CountyFIPS")[weather_cols].apply(
    lambda x: x.isnull().mean().mean()
).reset_index()
county_missing.columns = ["CountyFIPS", "weather_missing_rate"]

# Flag counties with more than 50% missing
high_missing = county_missing[county_missing["weather_missing_rate"] > 0.5]

print(f"Total counties: {county_missing['CountyFIPS'].nunique()}")
print(f"Counties with >50% weather data missing: {len(high_missing)}")
print(f"Counties retained: {len(county_missing) - len(high_missing)}")
print("\nTop 10 counties with most missing weather data:")
print(county_missing.sort_values("weather_missing_rate", ascending=False).head(10))

# Remove counties with >50% missing weather data
counties_to_keep = county_missing[county_missing["weather_missing_rate"] <= 0.5]["CountyFIPS"]
merged_df_filtered = merged_df[merged_df["CountyFIPS"].isin(counties_to_keep)]

print(f"\nMerged df shape before: {merged_df.shape}")
print(f"Merged df shape after: {merged_df_filtered.shape}")
merged_df_filtered.head()

Total counties: 3143
Counties with >50% weather data missing: 2248
Counties retained: 895

Top 10 counties with most missing weather data:
     CountyFIPS  weather_missing_rate
0         01001                   1.0
2178      40093                   1.0
1897      37013                   1.0
1898      37015                   1.0
1899      37017                   1.0
1900      37019                   1.0
1903      37025                   1.0
1904      37027                   1.0
1905      37029                   1.0
1907      37033                   1.0

Merged df shape before: (20478, 51)
Merged df shape after: (6646, 51)


,year,StateAbbr,County name,CountyFIPS,BPHIGH,CASTHMA,COPD,MHLTH,PHLTH,SLEEP,...,pct_bachelors_plus,pct_graduate_degree,pct_less_than_hs,pct_white,pct_black,pct_asian,pct_hispanic,median_household_income,state,county
0,2013,AK,ANCHORAGE MUNICIPALITY ...,02020,28.691667,NaN,NaN,NaN,NaN,NaN,...,11.7,92.5,23.2,0.1,7.9,7.8,92.0,77454,2,20
1,2013,AL,JEFFERSON COUNTY ...,01073,41.893750,NaN,NaN,NaN,NaN,NaN,...,11.6,87.4,26.6,0.1,0.8,0.8,96.2,45429,1,73
2,2013,AL,MADISON COUNTY ...,01089,37.654348,NaN,NaN,NaN,NaN,NaN,...,14.5,90.0,21.4,0.1,2.3,2.2,95.4,58434,1,89
3,2013,AL,MOBILE COUNTY ...,01097,44.833333,NaN,NaN,NaN,NaN,NaN,...,7.0,83.9,33.0,0.0,1.4,1.4,97.5,43028,1,97
4,2013,AL,MONTGOMERY COUNTY ...,01101,39.792727,NaN,NaN,NaN,NaN,NaN,...,12.0,85.6,26.2,0.1,1.3,1.1,96.5,44790,1,101


In [7]:
# Export unique list of counties with names, states and FIPS code
unique_counties = (
    merged_df_filtered[["CountyFIPS", "County name", "StateAbbr"]]
    .drop_duplicates()
    .sort_values(["StateAbbr", "County name"])
    .reset_index(drop=True)
)

print(f"Total unique counties in final dataset: {len(unique_counties)}")
print(unique_counties)

unique_counties.to_csv("final_county_list.csv", index=False)
print("\nExported to final_county_list.csv")

Total unique counties in final dataset: 895
    CountyFIPS                                        County name StateAbbr
0        02013  ALEUTIANS EAST BOROUGH                        ...        AK
1        02016  ALEUTIANS WEST CENSUS AREA                    ...        AK
2        02020  ANCHORAGE MUNICIPALITY                        ...        AK
3        02050  BETHEL CENSUS AREA                            ...        AK
4        02060  BRISTOL BAY BOROUGH                           ...        AK
..         ...                                                ...       ...
890      56037  SWEETWATER COUNTY                             ...        WY
891      56039  TETON COUNTY                                  ...        WY
892      56041  UINTA COUNTY                                  ...        WY
893      56043  WASHAKIE COUNTY                               ...        WY
894      56045  WESTON COUNTY                                 ...        WY

[895 rows x 3 columns]

Exported to final_c

In [8]:
# import climate zone data
pf = pq.ParquetFile("climate_types_county_2015.parquet")
climate_zones = pf.read().to_pandas(ignore_metadata=True)
climate_zones.head()

,county,climate_type_short,climate_type_long,Af,Am,Aw,BWh,BWk,BSh,BSk,...,Dwa,Dwb,Dwc,Dwd,Dfa,Dfb,Dfc,Dfd,ET,EF
0,20175,BSk,"Arid, steppe, cold",0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,28141,Cfa,"Temperate, no dry season, hot summer",0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,36101,Dfb,"Cold, no dry season, warm summer",0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,50013,Dfb,"Cold, no dry season, warm summer",0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,05065,Cfa,"Temperate, no dry season, hot summer",0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# Check CountyFIPS key columns before merging
print("climate_zones columns:", climate_zones.columns.tolist())
print("Sample climate_zones county:", climate_zones["county"].head(5).tolist())
print("Sample merged_df_filtered CountyFIPS:", merged_df_filtered["CountyFIPS"].head(5).tolist())

# Standardise county key in climate_zones to 5-digit string
climate_zones["county"] = climate_zones["county"].astype(str).str.zfill(5)

# Merge climate_type_short into merged_df_filtered
merged_df_filtered = merged_df_filtered.merge(
    climate_zones[["county", "climate_type_short"]],
    left_on="CountyFIPS",
    right_on="county",
    how="left"
).drop(columns="county_y")  # drop duplicate key column after merge

print(f"\nShape after climate zone merge: {merged_df_filtered.shape}")
print(f"Missing climate_type_short: {merged_df_filtered['climate_type_short'].isna().sum()}")
print(f"Unique climate zones: {merged_df_filtered['climate_type_short'].unique()}")
merged_df_filtered.head()

climate_zones columns: ['county', 'climate_type_short', 'climate_type_long', 'Af', 'Am', 'Aw', 'BWh', 'BWk', 'BSh', 'BSk', 'Csa', 'Csb', 'Csc', 'Cwa', 'Cwb', 'Cwc', 'Cfa', 'Cfb', 'Cfc', 'Dsa', 'Dsb', 'Dsc', 'Dsd', 'Dwa', 'Dwb', 'Dwc', 'Dwd', 'Dfa', 'Dfb', 'Dfc', 'Dfd', 'ET', 'EF']
Sample climate_zones county: ['20175', '28141', '36101', '50013', '05065']
Sample merged_df_filtered CountyFIPS: ['02020', '01073', '01089', '01097', '01101']

Shape after climate zone merge: (6646, 52)
Missing climate_type_short: 0
Unique climate zones: <ArrowStringArray>
[ 'ET', 'Cfa', 'BWh', 'BSk', 'Csb', 'Csa', 'Dfc', 'Dfa',  'Am',  'Aw', 'Dfb',
 'BWk', 'BSh', 'Dsb', 'Dsc', 'Cfb',  'Af', 'Dwb', 'Dwa']
Length: 19, dtype: str


,year,StateAbbr,County name,CountyFIPS,BPHIGH,CASTHMA,COPD,MHLTH,PHLTH,SLEEP,...,pct_graduate_degree,pct_less_than_hs,pct_white,pct_black,pct_asian,pct_hispanic,median_household_income,state,county_x,climate_type_short
0,2013,AK,ANCHORAGE MUNICIPALITY ...,02020,28.691667,NaN,NaN,NaN,NaN,NaN,...,92.5,23.2,0.1,7.9,7.8,92.0,77454,2,20,ET
1,2013,AL,JEFFERSON COUNTY ...,01073,41.893750,NaN,NaN,NaN,NaN,NaN,...,87.4,26.6,0.1,0.8,0.8,96.2,45429,1,73,Cfa
2,2013,AL,MADISON COUNTY ...,01089,37.654348,NaN,NaN,NaN,NaN,NaN,...,90.0,21.4,0.1,2.3,2.2,95.4,58434,1,89,Cfa
3,2013,AL,MOBILE COUNTY ...,01097,44.833333,NaN,NaN,NaN,NaN,NaN,...,83.9,33.0,0.0,1.4,1.4,97.5,43028,1,97,Cfa
4,2013,AL,MONTGOMERY COUNTY ...,01101,39.792727,NaN,NaN,NaN,NaN,NaN,...,85.6,26.2,0.1,1.3,1.1,96.5,44790,1,101,Cfa


In [10]:
# Define output path to data folder in parent project-krasss directory
output_path = os.path.join("..", "data", "merged_final.csv")

# Export final merged and filtered dataset
merged_df_filtered.to_csv(output_path, index=False)

print(f"Exported {len(merged_df_filtered)} rows to {output_path}")
print(f"Columns: {merged_df_filtered.columns.tolist()}")
print(f"Unique counties: {merged_df_filtered['CountyFIPS'].nunique()}")
print(f"Unique years: {sorted(merged_df_filtered['year'].unique())}")

Exported 6646 rows to ../data/merged_final.csv
Columns: ['year', 'StateAbbr', 'County name', 'CountyFIPS', 'BPHIGH', 'CASTHMA', 'COPD', 'MHLTH', 'PHLTH', 'SLEEP', 'STROKE', 'STATION', 'NAME_x', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'CLDD', 'DT100', 'DT32', 'DX32', 'DX70', 'DX90', 'EMNT', 'EMXT', 'HTDD', 'PRCP', 'TAVG', 'TMAX', 'TMIN', 'COUNTY', 'ST', 'Matched County_x', 'CountyFIPS_x', 'Matched County_y', 'CountyFIPS_y', 'Matched County', 'NAME_y', 'total_population', 'pct_female', 'pct_male', 'median_age', 'pct_bachelors_plus', 'pct_graduate_degree', 'pct_less_than_hs', 'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic', 'median_household_income', 'state', 'county_x', 'climate_type_short']
Unique counties: 895
Unique years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


In [11]:
# Clean County name column by stripping extra whitespace
merged_df_filtered["County name"] = merged_df_filtered["County name"].str.strip().str.replace(r'\s+', ' ', regex=True)

# Verify
print(merged_df_filtered["County name"].head(10).tolist())

['ANCHORAGE MUNICIPALITY', 'JEFFERSON COUNTY', 'MADISON COUNTY', 'MOBILE COUNTY', 'MONTGOMERY COUNTY', 'SHELBY COUNTY', 'TUSCALOOSA COUNTY', 'BENTON COUNTY', 'CRAIGHEAD COUNTY', 'PULASKI COUNTY']


In [12]:
# Remove duplicate and unnecessary columns
cols_to_drop = ['DT100', 'CountyFIPS_y', 'Matched County_y', 'CountyFIPS_x', 'Matched County_x']

# Only drop columns that actually exist in the dataframe
cols_to_drop_existing = [col for col in cols_to_drop if col in merged_df_filtered.columns]
merged_df_filtered = merged_df_filtered.drop(columns=cols_to_drop_existing)

print(f"Dropped columns: {cols_to_drop_existing}")
print(f"Remaining shape: {merged_df_filtered.shape}")

Dropped columns: ['DT100', 'CountyFIPS_y', 'Matched County_y', 'CountyFIPS_x', 'Matched County_x']
Remaining shape: (6646, 47)


In [13]:
# Apply log1p transformation to skewed weather columns in the final dataset
log_cols = ['CLDD', 'HTDD', 'DX90', 'DX32', 'DT32']

for col in log_cols:
    merged_df_filtered[col] = np.log1p(merged_df_filtered[col])

# Verify
print("Log-transformed columns:", log_cols)
print(merged_df_filtered[log_cols].describe().round(3))

# Re-export the updated dataset
output_path = os.path.join("..", "data", "merged_final_transformed.csv")
merged_df_filtered.to_csv(output_path, index=False)
print(f"\nRe-exported {len(merged_df_filtered)} rows to {output_path}")

Log-transformed columns: ['CLDD', 'HTDD', 'DX90', 'DX32', 'DT32']
           CLDD      HTDD      DX90      DX32      DT32
count  6125.000  6111.000  6162.000  6162.000  6155.000
mean      6.272     7.632     3.308     2.182     4.107
std       1.313     0.912     1.340     1.612     1.336
min       0.000     0.000     0.000     0.000     0.000
25%       5.908     7.261     2.565     0.000     3.689
50%       6.476     7.888     3.611     2.485     4.615
75%       7.073     8.188     4.369     3.631     4.997
max       8.190     9.276     5.403     5.489     5.720

Re-exported 6646 rows to ../data/merged_final_transformed.csv
